In [ ]:
# Make GETM Transect Animation (xarray + matplotlib)

# This notebook extracts a fixed-longitude transect from a GETM/ERSEM NetCDF file

# and builds an animated section plot over time.

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [11]:
# ------------------------------ User settings ------------------------------
NETCDF_PATH = Path("/export/lv9/user/qzhan/model_output/active_runs/dws_200m/dws_200m/32x32/2015/3d_merged_1d.nc")
OUTPUT_GIF = Path("/export/lv9/user/qzhan/model_output/active_runs/movie/transect_temp.gif")

PLOTVAR = "temp"
INDEX_LON_TRANSECT = 388
LAT_START = 240
LAT_STOP = 392
TIME_STEP = 4

VMIN = 3
VMAX = 6
CMAP = "RdBu_r"  # dark blue (cold) -> red (warm)

# ------------------------------ Helpers -----------------------------------
def _find_time_dim(da: xr.DataArray) -> str:
    for dim in da.dims:
        if "time" in dim.lower():
            return dim
    raise ValueError(f"No time-like dimension found in {da.dims}")


def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str:
    candidates = ("z", "sigma", "layer", "level", "lev", "depth", "nmesh2_layer_3d")
    for dim in da.dims:
        if dim == time_dim:
            continue
        if any(key in dim.lower() for key in candidates):
            return dim
    # Fallback: assume second dimension is vertical for 4D fields [time, z, y, x]
    if len(da.dims) >= 3:
        return da.dims[1]
    raise ValueError(f"No vertical-like dimension found in {da.dims}")


def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(
            f"Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}"
        )
    return remain[0], remain[1]


def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None


def extract_transect(
    ds: xr.Dataset,
    var_name: str,
    i_lon: int,
    lat_start: int,
    lat_stop: int,
    lon_name_candidates: tuple[str, ...] = ("lonc", "lon", "longitude"),
    lat_name_candidates: tuple[str, ...] = ("latc", "lat", "latitude"),
) -> dict[str, xr.DataArray]:
    da = ds[var_name]
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)
    y_dim, x_dim = _find_xy_dims(da, time_dim, z_dim)

    transect = da.isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})

    lat_name = _pick_coord_name(ds, lat_name_candidates)
    if lat_name is None:
        raise KeyError(f"Could not find latitude variable among {lat_name_candidates}")

    lat_da = ds[lat_name]
    if y_dim in lat_da.dims and x_dim in lat_da.dims:
        lat_line = lat_da.isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})
    elif y_dim in lat_da.dims:
        lat_line = lat_da.isel({y_dim: slice(lat_start, lat_stop)})
    else:
        raise ValueError(f"Latitude variable '{lat_name}' does not match transect dims")

    if "h" in ds.variables:
        h = ds["h"].isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})
        if z_dim in h.dims:
            depth = -h.cumsum(dim=z_dim)
        else:
            depth = None
    else:
        depth = None

    return {
        "transect": transect,
        "lat_line": lat_line,
        "depth": depth,
        "time_dim": transect.dims[0],
        "z_dim": z_dim,
        "y_dim": y_dim,
    }

In [12]:
ds = xr.open_dataset(NETCDF_PATH, decode_times=True)
pack = extract_transect(
    ds=ds,
    var_name=PLOTVAR,
    i_lon=INDEX_LON_TRANSECT,
    lat_start=LAT_START,
    lat_stop=LAT_STOP,
    lon_name_candidates=("lonc", "lon", "longitude"),
    lat_name_candidates=("latc", "lat", "latitude"),
)

transect = pack["transect"]
lat_line = pack["lat_line"]
depth = pack["depth"]
time_dim = pack["time_dim"]
z_dim = pack["z_dim"]
y_dim = pack["y_dim"]

times = transect[time_dim]
frame_indices = np.arange(0, transect.sizes[time_dim], TIME_STEP)

if len(frame_indices) == 0:
    raise ValueError("No frames selected. Check TIME_STEP and input time dimension length.")

fig, ax = plt.subplots(figsize=(10, 5))
plt.close(fig)  # Avoid duplicate static display in notebooks

# Create one persistent colorbar for the full animation range.
norm = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label(f"{PLOTVAR}")


def _get_xy_and_values(frame_idx: int):
    frame = transect.isel({time_dim: frame_idx})
    frame = frame.transpose(z_dim, y_dim)

    x2d = np.broadcast_to(np.asarray(lat_line)[None, :], frame.shape)

    if depth is not None:
        z2d = depth.isel({time_dim: frame_idx}).transpose(z_dim, y_dim).values
    else:
        # Fallback depth coordinate if layer thickness is not available
        z_levels = -np.arange(frame.shape[0], dtype=float)
        z2d = np.broadcast_to(z_levels[:, None], frame.shape)

    return x2d, z2d, frame.values


def update(frame_number: int):
    frame_idx = int(frame_indices[frame_number])
    x2d, z2d, values = _get_xy_and_values(frame_idx)

    ax.clear()
    mesh = ax.pcolormesh(
        x2d,
        z2d,
        values,
        shading="auto",
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
    )

    timestamp = np.asarray(times.isel({time_dim: frame_idx}).values)
    ax.set_title(f"{PLOTVAR} transect | {timestamp}")
    ax.set_xlabel("Latitude")
    ax.set_ylabel("Depth (m)")
    ax.set_ylim(-16, 1)

    # Legend-style metadata box for animation context
    ax.plot([], [], " ", label=f"Range: [{VMIN}, {VMAX}]")
    ax.plot([], [], " ", label=f"Lon index: {INDEX_LON_TRANSECT}")
    ax.plot([], [], " ", label=f"Lat idx: {LAT_START}:{LAT_STOP}")
    ax.plot([], [], " ", label=f"Frame step: {TIME_STEP}")
    ax.legend(loc="lower left", framealpha=0.85, facecolor="white")

    return (mesh,)


ani = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=150,
    blit=False,
    repeat=False,
  )

HTML(ani.to_jshtml())

OUTPUT_GIF.parent.mkdir(parents=True, exist_ok=True)
ani.save(OUTPUT_GIF, writer=PillowWriter(fps=6))
print(f"Saved animation to: {OUTPUT_GIF}")

ds.close()

Saved animation to: /export/lv9/user/qzhan/model_output/active_runs/movie/transect_temp.gif


In [14]:
# Make animation of transect elevation over time to inspect tidal water-column motion.
# Bathymetry stays fixed; the free surface moves with the surface elevation signal.

ELEVATION_VAR_CANDIDATES = ("elev", "eta", "eta_out", "elevation", "ssh", "zeta")
BATHY_VAR_CANDIDATES = ("bathymetry",)
OUTPUT_ELEV_GIF = OUTPUT_GIF.with_name(f"{OUTPUT_GIF.stem}_elevation.gif")


def _pick_existing_var(ds: xr.Dataset, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"Could not find any of these variables: {candidates}")


def _find_horizontal_dims(da: xr.DataArray, exclude: tuple[str, ...] = ()) -> tuple[str, str]:
    dims = [dim for dim in da.dims if dim not in exclude]
    if len(dims) != 2:
        raise ValueError(f"Expected 2 horizontal dims, got {dims} from {da.dims}")
    return dims[0], dims[1]


ds_elev = xr.open_dataset(NETCDF_PATH, decode_times=True)

elev_name = _pick_existing_var(ds_elev, ELEVATION_VAR_CANDIDATES)
bathy_name = _pick_existing_var(ds_elev, BATHY_VAR_CANDIDATES)
lat_name = _pick_coord_name(ds_elev, ("latc", "lat", "latitude"))
if lat_name is None:
    raise KeyError("Could not find a latitude coordinate for the transect.")

elev_da = ds_elev[elev_name]
elev_time_dim = _find_time_dim(elev_da)
elev_y_dim, elev_x_dim = _find_horizontal_dims(elev_da, exclude=(elev_time_dim,))

elev_line = elev_da.isel({
    elev_x_dim: INDEX_LON_TRANSECT,
    elev_y_dim: slice(LAT_START, LAT_STOP),
}).transpose(elev_time_dim, elev_y_dim)

bathy_da = ds_elev[bathy_name]
bathy_y_dim, bathy_x_dim = _find_horizontal_dims(bathy_da)
bathy_line = bathy_da.isel({
    bathy_x_dim: INDEX_LON_TRANSECT,
    bathy_y_dim: slice(LAT_START, LAT_STOP),
})

lat_da = ds_elev[lat_name]
if elev_y_dim in lat_da.dims and elev_x_dim in lat_da.dims:
    lat_line_elev = lat_da.isel({
        elev_x_dim: INDEX_LON_TRANSECT,
        elev_y_dim: slice(LAT_START, LAT_STOP),
    })
elif elev_y_dim in lat_da.dims:
    lat_line_elev = lat_da.isel({elev_y_dim: slice(LAT_START, LAT_STOP)})
else:
    raise ValueError(f"Latitude variable '{lat_name}' does not match transect dims")

lat_values = np.asarray(lat_line_elev.values)
bottom_values = np.where(np.asarray(bathy_line.values) <= -9990, np.nan, -np.asarray(bathy_line.values))
times_elev = elev_line[elev_time_dim]
frame_indices_elev = np.arange(0, elev_line.sizes[elev_time_dim], TIME_STEP)

if len(frame_indices_elev) == 0:
    raise ValueError("No elevation frames selected. Check TIME_STEP and input time dimension length.")

surface_min = float(np.nanmin(np.asarray(elev_line.values)))
surface_max = float(np.nanmax(np.asarray(elev_line.values)))
ymin = min(-16.0, float(np.nanmin(bottom_values)))
ymax = max(1.0, surface_max + 0.2)

fig_elev, ax_elev = plt.subplots(figsize=(10, 4.5))
plt.close(fig_elev)


def update_elev(frame_number: int):
    frame_idx = int(frame_indices_elev[frame_number])
    surface_values = np.asarray(elev_line.isel({elev_time_dim: frame_idx}).values)

    ax_elev.clear()
    ax_elev.fill_between(
        lat_values,
        bottom_values,
        surface_values,
        color="steelblue",
        alpha=0.45,
        label="Water column",
    )
    ax_elev.plot(lat_values, bottom_values, color="black", linewidth=1.5, label="Bathymetry")
    ax_elev.plot(lat_values, surface_values, color="royalblue", linewidth=2.0, label=elev_name)

    timestamp = np.asarray(times_elev.isel({elev_time_dim: frame_idx}).values)
    ax_elev.set_title(f"Transect elevation | {timestamp}")
    ax_elev.set_xlabel("Latitude")
    ax_elev.set_ylabel("Elevation / depth (m)")
    ax_elev.set_ylim(ymin, ymax)
    ax_elev.grid(True, alpha=0.3)
    ax_elev.legend(loc="lower left", framealpha=0.9, facecolor="white")

    return tuple(ax_elev.lines)


ani_elev = FuncAnimation(
    fig_elev,
    update_elev,
    frames=len(frame_indices_elev),
    interval=150,
    blit=False,
    repeat=False,
  )

elev_html = HTML(ani_elev.to_jshtml())
OUTPUT_ELEV_GIF.parent.mkdir(parents=True, exist_ok=True)
ani_elev.save(OUTPUT_ELEV_GIF, writer=PillowWriter(fps=6))
print(f"Saved elevation animation to: {OUTPUT_ELEV_GIF}")
ds_elev.close()

elev_html   

Saved elevation animation to: /export/lv9/user/qzhan/model_output/active_runs/movie/transect_temp_elevation.gif


In [18]:
# Make animation of transect elevation over time, with the water column colored by temperature.
# Bathymetry stays fixed; the free surface moves with the surface elevation signal.

TEMP_VAR = "temp"
OUTPUT_TEMP_ELEV_GIF = OUTPUT_GIF.with_name(f"{OUTPUT_GIF.stem}_elevation_temperature.gif")

ds_fill = xr.open_dataset(NETCDF_PATH, decode_times=True)

temp_pack = extract_transect(
    ds=ds_fill,
    var_name=TEMP_VAR,
    i_lon=INDEX_LON_TRANSECT,
    lat_start=LAT_START,
    lat_stop=LAT_STOP,
    lon_name_candidates=("lonc", "lon", "longitude"),
    lat_name_candidates=("latc", "lat", "latitude"),
)

temp_transect = temp_pack["transect"]
temp_time_dim = temp_pack["time_dim"]
temp_z_dim = temp_pack["z_dim"]
temp_y_dim = temp_pack["y_dim"]

elev_name = _pick_existing_var(ds_fill, ELEVATION_VAR_CANDIDATES)
bathy_name = _pick_existing_var(ds_fill, BATHY_VAR_CANDIDATES)

elev_da = ds_fill[elev_name]
elev_time_dim = _find_time_dim(elev_da)
elev_y_dim, elev_x_dim = _find_horizontal_dims(elev_da, exclude=(elev_time_dim,))
elev_line = elev_da.isel({
    elev_x_dim: INDEX_LON_TRANSECT,
    elev_y_dim: slice(LAT_START, LAT_STOP),
}).transpose(elev_time_dim, elev_y_dim)

bathy_da = ds_fill[bathy_name]
bathy_y_dim, bathy_x_dim = _find_horizontal_dims(bathy_da)
bathy_line = bathy_da.isel({
    bathy_x_dim: INDEX_LON_TRANSECT,
    bathy_y_dim: slice(LAT_START, LAT_STOP),
})

lat_line_fill = temp_pack["lat_line"]
lat_values_fill = np.asarray(lat_line_fill.values, dtype=float)
bottom_values_fill = np.where(
    np.asarray(bathy_line.values) <= -9990,
    np.nan,
    -np.asarray(bathy_line.values, dtype=float),
)

valid_static_idx = np.flatnonzero(
    np.isfinite(lat_values_fill) & np.isfinite(bottom_values_fill)
    )
if valid_static_idx.size < 2:
    raise ValueError("Not enough valid transect columns after removing invalid bathymetry points.")

temp_transect = temp_transect.isel({temp_y_dim: valid_static_idx})
elev_line = elev_line.isel({elev_y_dim: valid_static_idx})
lat_values_fill = lat_values_fill[valid_static_idx]
bottom_values_fill = bottom_values_fill[valid_static_idx]

times_fill = temp_transect[temp_time_dim]
frame_indices_fill = np.arange(0, temp_transect.sizes[temp_time_dim], TIME_STEP)

if len(frame_indices_fill) == 0:
    raise ValueError("No temperature-elevation frames selected. Check TIME_STEP and input time dimension length.")

if temp_transect.sizes[temp_z_dim] < 1:
    raise ValueError("Temperature transect must have at least one vertical layer.")

sigma_edges = np.linspace(0.0, 1.0, temp_transect.sizes[temp_z_dim] + 1)[:, None]

fig_fill, ax_fill = plt.subplots(figsize=(10, 5))
plt.close(fig_fill)

norm_fill = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm_fill = plt.cm.ScalarMappable(norm=norm_fill, cmap=CMAP)
sm_fill.set_array([])
cbar_fill = fig_fill.colorbar(sm_fill, ax=ax_fill, fraction=0.035, pad=0.03)
cbar_fill.set_label(f"{TEMP_VAR}")


def _compute_edges(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        raise ValueError("Need at least two points to build cell edges.")
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (values[:-1] + values[1:])
    edges[0] = values[0] - 0.5 * (values[1] - values[0])
    edges[-1] = values[-1] + 0.5 * (values[-1] - values[-2])
    return edges


def _make_vertical_edges(surface_values: np.ndarray, bottom_values: np.ndarray) -> np.ndarray:
    bottom_edges = _compute_edges(bottom_values)
    surface_edges = _compute_edges(surface_values)
    water_column_height = surface_edges - bottom_edges
    return bottom_edges[None, :] + sigma_edges * water_column_height[None, :]


def update_fill(frame_number: int):
    frame_idx = int(frame_indices_fill[frame_number])
    temp_values = np.asarray(
        temp_transect.isel({temp_time_dim: frame_idx}).transpose(temp_z_dim, temp_y_dim).values,
        dtype=float,
    )
    surface_values = np.asarray(elev_line.isel({elev_time_dim: frame_idx}).values, dtype=float)

    valid_frame = np.isfinite(surface_values) & np.isfinite(bottom_values_fill)
    if valid_frame.sum() < 2:
        raise ValueError("Not enough valid columns in this frame to build the temperature mesh.")

    lat_plot = lat_values_fill[valid_frame]
    bottom_plot = bottom_values_fill[valid_frame]
    surface_plot = surface_values[valid_frame]
    temp_plot = np.ma.masked_invalid(temp_values[:, valid_frame])

    x_edges_plot = _compute_edges(lat_plot)
    x_edges_2d = np.broadcast_to(x_edges_plot[None, :], (temp_plot.shape[0] + 1, x_edges_plot.size))
    z_edges_2d = _make_vertical_edges(surface_plot, bottom_plot)

    ax_fill.clear()
    mesh = ax_fill.pcolormesh(
        x_edges_2d,
        z_edges_2d,
        temp_plot,
        shading="auto",
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
    )
    ax_fill.plot(lat_plot, bottom_plot, color="black", linewidth=1.2, label="Bathymetry")
    ax_fill.plot(lat_plot, surface_plot, color="white", linewidth=1.2, label=elev_name)

    timestamp = np.asarray(times_fill.isel({temp_time_dim: frame_idx}).values)
    ax_fill.set_title(f"Temperature-filled water column | {timestamp}")
    ax_fill.set_xlabel("Latitude")
    ax_fill.set_ylabel("Elevation / depth (m)")
    ax_fill.set_ylim(-16, 1)
    ax_fill.grid(True, alpha=0.25)
    ax_fill.legend(loc="lower left", framealpha=0.9, facecolor="white")

    return (mesh,)


ani_fill = FuncAnimation(
    fig_fill,
    update_fill,
    frames=len(frame_indices_fill),
    interval=150,
    blit=False,
    repeat=False,
  )

fill_html = HTML(ani_fill.to_jshtml())
OUTPUT_TEMP_ELEV_GIF.parent.mkdir(parents=True, exist_ok=True)
ani_fill.save(OUTPUT_TEMP_ELEV_GIF, writer=PillowWriter(fps=6))
print(f"Saved temperature-elevation animation to: {OUTPUT_TEMP_ELEV_GIF}")
ds_fill.close()

fill_html

Saved temperature-elevation animation to: /export/lv9/user/qzhan/model_output/active_runs/movie/transect_temp_elevation_temperature.gif


In [21]:
# Make animation of transect elevation over time, with the water column colored by PO4 concentration.
# Bathymetry stays fixed; the free surface moves with the surface elevation signal.

PO4_VAR = "N1p"  # Replace if your PO4 variable uses a different name
OUTPUT_PO4_ELEV_GIF = OUTPUT_GIF.with_name(f"{OUTPUT_GIF.stem}_elevation_PO4.gif")

ds_fill = xr.open_dataset(NETCDF_PATH, decode_times=True)

po4_pack = extract_transect(
    ds=ds_fill,
    var_name=PO4_VAR,
    i_lon=INDEX_LON_TRANSECT,
    lat_start=LAT_START,
    lat_stop=LAT_STOP,
    lon_name_candidates=("lonc", "lon", "longitude"),
    lat_name_candidates=("latc", "lat", "latitude"),
)

po4_transect = po4_pack["transect"]
po4_time_dim = po4_pack["time_dim"]
po4_z_dim = po4_pack["z_dim"]
po4_y_dim = po4_pack["y_dim"]

elev_name = _pick_existing_var(ds_fill, ELEVATION_VAR_CANDIDATES)
bathy_name = _pick_existing_var(ds_fill, BATHY_VAR_CANDIDATES)

elev_da = ds_fill[elev_name]
elev_time_dim = _find_time_dim(elev_da)
elev_y_dim, elev_x_dim = _find_horizontal_dims(elev_da, exclude=(elev_time_dim,))
elev_line = elev_da.isel({
    elev_x_dim: INDEX_LON_TRANSECT,
    elev_y_dim: slice(LAT_START, LAT_STOP),
}).transpose(elev_time_dim, elev_y_dim)

bathy_da = ds_fill[bathy_name]
bathy_y_dim, bathy_x_dim = _find_horizontal_dims(bathy_da)
bathy_line = bathy_da.isel({
    bathy_x_dim: INDEX_LON_TRANSECT,
    bathy_y_dim: slice(LAT_START, LAT_STOP),
})

lat_line_fill = po4_pack["lat_line"]
lat_values_fill = np.asarray(lat_line_fill.values, dtype=float)
bottom_values_fill = np.where(
    np.asarray(bathy_line.values) <= -9990,
    np.nan,
    -np.asarray(bathy_line.values, dtype=float),
)

valid_static_idx = np.flatnonzero(
    np.isfinite(lat_values_fill) & np.isfinite(bottom_values_fill)
    )
if valid_static_idx.size < 2:
    raise ValueError("Not enough valid transect columns after removing invalid bathymetry points.")

po4_transect = po4_transect.isel({po4_y_dim: valid_static_idx})
elev_line = elev_line.isel({elev_y_dim: valid_static_idx})
lat_values_fill = lat_values_fill[valid_static_idx]
bottom_values_fill = bottom_values_fill[valid_static_idx]

times_fill = po4_transect[po4_time_dim]
frame_indices_fill = np.arange(0, po4_transect.sizes[po4_time_dim], TIME_STEP)

if len(frame_indices_fill) == 0:
    raise ValueError("No PO4-elevation frames selected. Check TIME_STEP and input time dimension length.")

if po4_transect.sizes[po4_z_dim] < 1:
    raise ValueError("PO4 transect must have at least one vertical layer.")

po4_min = float(np.nanmin(np.asarray(po4_transect.values, dtype=float)))
po4_max = float(np.nanmax(np.asarray(po4_transect.values, dtype=float)))
if not np.isfinite(po4_min) or not np.isfinite(po4_max):
    raise ValueError("PO4 transect has no finite values to define a plotting range.")
if po4_max <= po4_min:
    po4_max = po4_min + 1e-12

sigma_edges = np.linspace(0.0, 1.0, po4_transect.sizes[po4_z_dim] + 1)[:, None]

fig_fill, ax_fill = plt.subplots(figsize=(10, 5))
plt.close(fig_fill)

norm_fill = plt.Normalize(vmin=po4_min, vmax=po4_max)
sm_fill = plt.cm.ScalarMappable(norm=norm_fill, cmap=CMAP)
sm_fill.set_array([])
cbar_fill = fig_fill.colorbar(sm_fill, ax=ax_fill, fraction=0.035, pad=0.03)
cbar_fill.set_label(f"{PO4_VAR}")


def _compute_edges(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        raise ValueError("Need at least two points to build cell edges.")
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (values[:-1] + values[1:])
    edges[0] = values[0] - 0.5 * (values[1] - values[0])
    edges[-1] = values[-1] + 0.5 * (values[-1] - values[-2])
    return edges


def _make_vertical_edges(surface_values: np.ndarray, bottom_values: np.ndarray) -> np.ndarray:
    bottom_edges = _compute_edges(bottom_values)
    surface_edges = _compute_edges(surface_values)
    water_column_height = surface_edges - bottom_edges
    return bottom_edges[None, :] + sigma_edges * water_column_height[None, :]


def update_fill(frame_number: int):
    frame_idx = int(frame_indices_fill[frame_number])
    po4_values = np.asarray(
        po4_transect.isel({po4_time_dim: frame_idx}).transpose(po4_z_dim, po4_y_dim).values,
        dtype=float,
    )
    surface_values = np.asarray(elev_line.isel({elev_time_dim: frame_idx}).values, dtype=float)

    valid_frame = np.isfinite(surface_values) & np.isfinite(bottom_values_fill)
    if valid_frame.sum() < 2:
        raise ValueError("Not enough valid columns in this frame to build the PO4 mesh.")

    lat_plot = lat_values_fill[valid_frame]
    bottom_plot = bottom_values_fill[valid_frame]
    surface_plot = surface_values[valid_frame]
    po4_plot = np.ma.masked_invalid(po4_values[:, valid_frame])

    x_edges_plot = _compute_edges(lat_plot)
    x_edges_2d = np.broadcast_to(x_edges_plot[None, :], (po4_plot.shape[0] + 1, x_edges_plot.size))
    z_edges_2d = _make_vertical_edges(surface_plot, bottom_plot)

    ax_fill.clear()
    mesh = ax_fill.pcolormesh(
        x_edges_2d,
        z_edges_2d,
        po4_plot,
        shading="auto",
        cmap=CMAP,
        vmin=po4_min,
        vmax=po4_max,
    )
    ax_fill.plot(lat_plot, bottom_plot, color="black", linewidth=1.2, label="Bathymetry")
    ax_fill.plot(lat_plot, surface_plot, color="white", linewidth=1.2, label=elev_name)
    ax_fill.plot([], [], " ", label=f"PO4 range: [{po4_min:.3g}, {po4_max:.3g}]")

    timestamp = np.asarray(times_fill.isel({po4_time_dim: frame_idx}).values)
    ax_fill.set_title(f"PO4-filled water column | {timestamp}")
    ax_fill.set_xlabel("Latitude")
    ax_fill.set_ylabel("Elevation / depth (m)")
    ax_fill.set_ylim(-16, 1)
    ax_fill.grid(True, alpha=0.25)
    ax_fill.legend(loc="lower left", framealpha=0.9, facecolor="white")

    return (mesh,)


ani_fill = FuncAnimation(
    fig_fill,
    update_fill,
    frames=len(frame_indices_fill),
    interval=150,
    blit=False,
    repeat=False,
  )

fill_html = HTML(ani_fill.to_jshtml())
OUTPUT_PO4_ELEV_GIF.parent.mkdir(parents=True, exist_ok=True)
ani_fill.save(OUTPUT_PO4_ELEV_GIF, writer=PillowWriter(fps=6))
print(f"Saved PO4-elevation animation to: {OUTPUT_PO4_ELEV_GIF}")
ds_fill.close()

fill_html

Saved PO4-elevation animation to: /export/lv9/user/qzhan/model_output/active_runs/movie/transect_temp_elevation_PO4.gif


In [3]:

import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from scipy.interpolate import griddata
from pathlib import Path

# ── Output path ──────────────────────────────────────────────────────────────
OUTPUT_TIFF = Path("/export/lv9/user/qzhan/home/temp/topo_georef.tif")

# ── Pull arrays from dataset ──────────────────────────────────────────────────
lon = ds["lonc"].values          # (yc, xc)
lat = ds["latc"].values          # (yc, xc)
bathy = ds["bathymetry"].values  # (yc, xc)  – positive = water depth

# ── Build a regular lat/lon target grid ──────────────────────────────────────
lon_min, lon_max = float(lon.min()), float(lon.max())
lat_min, lat_max = float(lat.min()), float(lat.max())

# Keep roughly the same spatial resolution (~500 m ≈ 0.005 deg)
n_lon = int(np.round((lon_max - lon_min) / 0.005))
n_lat = int(np.round((lat_max - lat_min) / 0.005))
print(f"Target grid: {n_lon} × {n_lat}  (lon × lat)")

lon_grid = np.linspace(lon_min, lon_max, n_lon)
lat_grid = np.linspace(lat_min, lat_max, n_lat)
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# ── Interpolate bathymetry onto the regular grid ──────────────────────────────
points = np.column_stack([lon.ravel(), lat.ravel()])
values = bathy.ravel()

# mask out NaN source points
valid = np.isfinite(values)
bathy_reg = griddata(points[valid], values[valid],
                     (lon2d, lat2d), method="linear")

# ── Write GeoTIFF (WGS84) ────────────────────────────────────────────────────
transform = from_bounds(lon_min, lat_min, lon_max, lat_max, n_lon, n_lat)
crs = CRS.from_epsg(4326)   # WGS84 geographic

bathy_reg_flipped = np.flipud(bathy_reg)  # rasterio expects top-row = north

with rasterio.open(
    OUTPUT_TIFF,
    "w",
    driver="GTiff",
    height=n_lat,
    width=n_lon,
    count=1,
    dtype=rasterio.float32,
    crs=crs,
    transform=transform,
    nodata=np.nan,
) as dst:
    dst.write(bathy_reg_flipped.astype(np.float32), 1)
    dst.update_tags(description="Bathymetry from GETM topo.nc, WGS84 georeferenced")

print(f"Saved → {OUTPUT_TIFF}")


Target grid: 551 × 251  (lon × lat)
Saved → /export/lv9/user/qzhan/home/temp/topo_georef.tif


In [ ]:
# Make animation of the surface layer over the full model domain.

SURFACE_LAYER_INDEX = 10  # 10 for the top layer, and 0 for the bottom layer
OUTPUT_SURFACE_GIF = OUTPUT_GIF.with_name(f"{OUTPUT_GIF.stem}_surface_domain.gif")

ds_surface = xr.open_dataset(NETCDF_PATH, decode_times=True)

surface_da = ds_surface[PLOTVAR]
surface_time_dim = _find_time_dim(surface_da)
surface_z_dim = _find_vertical_dim(surface_da, surface_time_dim)
surface_y_dim, surface_x_dim = _find_xy_dims(surface_da, surface_time_dim, surface_z_dim)

if SURFACE_LAYER_INDEX >= surface_da.sizes[surface_z_dim] or SURFACE_LAYER_INDEX < -surface_da.sizes[surface_z_dim]:
    raise IndexError(
        f"SURFACE_LAYER_INDEX={SURFACE_LAYER_INDEX} is out of bounds for '{surface_z_dim}' "
        f"with size {surface_da.sizes[surface_z_dim]}"
    )

surface_field = surface_da.isel({surface_z_dim: SURFACE_LAYER_INDEX})
times_surface = surface_field[surface_time_dim]
frame_indices_surface = np.arange(0, surface_field.sizes[surface_time_dim], TIME_STEP)

if len(frame_indices_surface) == 0:
    raise ValueError("No surface frames selected. Check TIME_STEP and input time dimension length.")

lon_name_surface = _pick_coord_name(ds_surface, ("lonc", "lon", "longitude"))
lat_name_surface = _pick_coord_name(ds_surface, ("latc", "lat", "latitude"))
if lon_name_surface is None or lat_name_surface is None:
    raise KeyError("Could not find both longitude and latitude coordinate variables.")

lon_da_surface = ds_surface[lon_name_surface]
lat_da_surface = ds_surface[lat_name_surface]

# Prefer georeferenced plotting, but fall back to index-space if coordinates contain NaN/Inf.
use_geo_coords = False
x_plot = None
y_plot = None
x_label = "X index"
y_label = "Y index"

if (
    surface_y_dim in lon_da_surface.dims
    and surface_x_dim in lon_da_surface.dims
    and surface_y_dim in lat_da_surface.dims
    and surface_x_dim in lat_da_surface.dims
):
    lon2d = np.asarray(lon_da_surface.values, dtype=float)
    lat2d = np.asarray(lat_da_surface.values, dtype=float)
    if np.isfinite(lon2d).all() and np.isfinite(lat2d).all():
        x_plot = lon2d
        y_plot = lat2d
        x_label = "Longitude"
        y_label = "Latitude"
        use_geo_coords = True
elif surface_x_dim in lon_da_surface.dims and surface_y_dim in lat_da_surface.dims:
    lon1d = np.asarray(lon_da_surface.values, dtype=float)
    lat1d = np.asarray(lat_da_surface.values, dtype=float)
    if np.isfinite(lon1d).all() and np.isfinite(lat1d).all():
        x_plot, y_plot = np.meshgrid(lon1d, lat1d)
        x_label = "Longitude"
        y_label = "Latitude"
        use_geo_coords = True

ny = surface_field.sizes[surface_y_dim]
nx = surface_field.sizes[surface_x_dim]

if x_plot is None or y_plot is None:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))
    print("Warning: lon/lat contain non-finite values; plotting in grid index space.")

if INDEX_LON_TRANSECT < 0 or INDEX_LON_TRANSECT >= nx:
    raise IndexError(
        f"INDEX_LON_TRANSECT={INDEX_LON_TRANSECT} is out of bounds for x-dimension size {nx}"
    )

transect_y_start = max(0, LAT_START)
transect_y_stop = min(ny, LAT_STOP)
if transect_y_stop - transect_y_start < 2:
    raise ValueError(
        f"Transect y-range [{LAT_START}:{LAT_STOP}] leaves fewer than 2 points in the surface grid."
    )

draw_transect_line = True
if use_geo_coords:
    tx = np.asarray(x_plot[transect_y_start:transect_y_stop, INDEX_LON_TRANSECT], dtype=float)
    ty = np.asarray(y_plot[transect_y_start:transect_y_stop, INDEX_LON_TRANSECT], dtype=float)
    valid_transect = np.isfinite(tx) & np.isfinite(ty)
    if valid_transect.sum() < 2:
        draw_transect_line = False
        transect_line_x = None
        transect_line_y = None
        print("Warning: Transect line has too few finite lon/lat points to draw.")
    else:
        transect_line_x = tx[valid_transect]
        transect_line_y = ty[valid_transect]
else:
    transect_line_x = np.full(transect_y_stop - transect_y_start, float(INDEX_LON_TRANSECT))
    transect_line_y = np.arange(transect_y_start, transect_y_stop, dtype=float)

fig_surface, ax_surface = plt.subplots(figsize=(8, 7))
plt.close(fig_surface)

norm_surface = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm_surface = plt.cm.ScalarMappable(norm=norm_surface, cmap=CMAP)
sm_surface.set_array([])
cbar_surface = fig_surface.colorbar(sm_surface, ax=ax_surface, fraction=0.035, pad=0.03)
cbar_surface.set_label(f"{PLOTVAR} (surface layer)")


def update_surface(frame_number: int):
    frame_idx = int(frame_indices_surface[frame_number])
    frame2d = np.asarray(
        surface_field.isel({surface_time_dim: frame_idx}).transpose(surface_y_dim, surface_x_dim).values,
        dtype=float,
    )

    ax_surface.clear()
    mesh = ax_surface.pcolormesh(
        x_plot,
        y_plot,
        np.ma.masked_invalid(frame2d),
        shading="auto",
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
    )

    if draw_transect_line:
        ax_surface.plot(
            transect_line_x,
            transect_line_y,
            linestyle="--",
            color="yellow",
            linewidth=2.0,
        )

    timestamp = np.asarray(times_surface.isel({surface_time_dim: frame_idx}).values)
    ax_surface.set_title(f"{PLOTVAR} surface layer | {timestamp}")
    ax_surface.set_xlabel(x_label)
    ax_surface.set_ylabel(y_label)
    if use_geo_coords:
        ax_surface.set_aspect("equal", adjustable="box")
    ax_surface.grid(True, alpha=0.2)

    return (mesh,)


ani_surface = FuncAnimation(
    fig_surface,
    update_surface,
    frames=len(frame_indices_surface),
    interval=150,
    blit=False,
    repeat=False,
  )

surface_html = HTML(ani_surface.to_jshtml())
OUTPUT_SURFACE_GIF.parent.mkdir(parents=True, exist_ok=True)
ani_surface.save(OUTPUT_SURFACE_GIF, writer=PillowWriter(fps=6))
print(f"Saved surface-domain animation to: {OUTPUT_SURFACE_GIF}")
ds_surface.close()

surface_html